# 03 · Segmentacion exploratoria de clientes

Ruta de continuacion con las seis fases CRISP-DM. Leemos los CSV publicos del repositorio.

## 1. Comprension del negocio

Buscamos grupos de clientes con historiales de compra parecidos. Describiremos cada grupo con sus medianas de recencia, frecuencia, gasto y variedad, y compararemos el perfil con la mediana general.

## 2. Comprension de los datos y EDA

Asociamos pedidos entregados, clientes seudónimos, registros de artículos con precio y categorías. Exploramos estados, fechas, precios y las distribuciones de variables agregadas por cliente.


In [ ]:
from urllib.error import HTTPError, URLError

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.cluster import Birch, KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

DATA_BASE_URL = "https://raw.githubusercontent.com/JuanParias29/ML_MiniWorkshop/main/data"
def load_dataset_csv(filename):
    url = f"{DATA_BASE_URL}/{filename}"
    try:
        return pd.read_csv(url)
    except (HTTPError, URLError) as error:
        raise RuntimeError(f"No se pudo cargar {filename} desde {url}. Compruebe la conexion a internet.") from error
orders = load_dataset_csv("olist_orders_dataset.csv")
customers = load_dataset_csv("olist_customers_dataset.csv")
items = load_dataset_csv("olist_order_items_dataset.csv")
products = load_dataset_csv("olist_products_dataset.csv")
print(f"Pedidos: {len(orders):,}; filas de articulos: {len(items):,}")


### EDA de tablas fuente

Inspeccionamos volumen por estado, compras por mes y precio por linea. Si la distribucion de precio tiene cola derecha, una transformacion logaritmica evita que algunos valores grandes dominen por escala; no implica que sean registros incorrectos.


In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
orders["order_status"].value_counts().plot.bar(ax=axes[0], color="slateblue")
axes[0].set_title("Pedidos por estado")
monthly_orders = orders.dropna(subset=["order_purchase_timestamp"]).set_index("order_purchase_timestamp").resample("MS").size()
monthly_orders.plot(ax=axes[1], color="darkorange")
axes[1].set_title("Pedidos por mes")
axes[1].set_ylabel("Pedidos")
np.log1p(items["price"].clip(lower=0)).plot.hist(bins=35, ax=axes[2], color="teal")
axes[2].set_title("Precio por línea — log(1+x)")
axes[2].set_xlabel("Precio transformado")
plt.tight_layout()
plt.show()
print(f"Cobertura de compra: {orders['order_purchase_timestamp'].min().date()} a {orders['order_purchase_timestamp'].max().date()}")
print(f"Precio BRL: mediana={items['price'].median():.2f}; media={items['price'].mean():.2f}; sesgo={items['price'].skew():.2f}")


## 3. Preparacion: variables por cliente y justificacion

Filtramos pedidos entregados con fecha y cliente válidos. Recencia son días desde la última compra; frecuencia cuenta pedidos distintos; gasto suma `price` (no incluye flete); variedad cuenta categorías conocidas. `customer_unique_id` solo agrupa los pedidos. Aplicamos `log1p` a las variables sesgadas y `StandardScaler` antes de comparar distancias. Maryani et al. ([2018](https://doi.org/10.1109/IAC.2018.8780570)) estudian RFM con K-Means; Milligan & Cooper ([1988](https://doi.org/10.1007/BF01897163)) analizan el efecto de la estandarización en clustering.


In [ ]:
orders = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="inner")
delivered = orders.loc[
    orders["order_status"].eq("delivered")
    & orders["order_purchase_timestamp"].notna()
    & orders["customer_unique_id"].notna()
].copy()
if delivered.empty:
    raise ValueError("No hay pedidos entregados con fecha y cliente validos.")
reference_date = delivered["order_purchase_timestamp"].max().normalize() + pd.Timedelta(days=1)
order_map = delivered[["order_id", "customer_unique_id", "order_purchase_timestamp"]]
purchases = items.merge(order_map, on="order_id", how="inner").merge(
    products[["product_id", "product_category_name"]], on="product_id", how="left"
)
customer_features = purchases.groupby("customer_unique_id").agg(
    frequency=("order_id", "nunique"),
    spend_brl=("price", "sum"),
    category_variety=("product_category_name", "nunique"),
    last_purchase=("order_purchase_timestamp", "max"),
)
customer_features["recency_days"] = (reference_date - customer_features.pop("last_purchase")).dt.total_seconds() / 86400
feature_columns = ["recency_days", "frequency", "spend_brl", "category_variety"]
customer_features = customer_features[feature_columns].replace([np.inf, -np.inf], np.nan).dropna()
if (customer_features[feature_columns] < 0).any().any():
    raise ValueError("log1p necesita variables no negativas.")
print(f"Clientes agregados: {len(customer_features):,}; fecha de referencia: {reference_date.date()}")
display(customer_features.describe().T.round(2))


### EDA de agregados

Los histogramas en log1p muestran la forma de recencia, frecuencia, gasto y variedad; la tabla esta en unidades originales. Media, mediana y sesgo permiten ver si hay cola derecha. La correlacion ayuda a detectar variables redundantes, pero no implica una relacion causal.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, column in zip(axes.flat, feature_columns):
    ax.hist(np.log1p(customer_features[column].clip(lower=0)), bins=40, color="teal", alpha=0.8)
    ax.set_title(f"{column} — log(1+x)")
    ax.set_xlabel("Valor transformado")
    ax.set_ylabel("Clientes")
plt.tight_layout()
plt.show()
summary = customer_features[feature_columns].agg(["median", "mean", "skew"]).T
summary.columns = ["mediana", "media", "sesgo"]
display(summary.round(2))
correlation = np.log1p(customer_features[feature_columns]).corr()
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(correlation, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(feature_columns)), feature_columns, rotation=45, ha="right")
ax.set_yticks(range(len(feature_columns)), feature_columns)
for row in range(len(feature_columns)):
    for column in range(len(feature_columns)):
        ax.text(column, row, f"{correlation.iloc[row, column]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlación tras log1p — exploratoria")
fig.colorbar(image, ax=ax, label="Correlacion")
plt.tight_layout()
plt.show()


## 4. Modelado: comparar tres algoritmos

Probamos K-Means, mezcla gaussiana (GMM) y Birch con 2, 3 o 4 grupos. Comparamos silhouette (más alto indica grupos más compactos y separados) y Davies–Bouldin (más bajo es preferible) en la misma muestra reproducible. Rousseeuw ([1987](https://doi.org/10.1016/0377-0427(87)90125-7)) presenta silhouette como medida interna de agrupamiento. Elegimos por silhouette y ajustamos el modelo seleccionado a todos los clientes.


In [ ]:
X = customer_features[feature_columns]
selection_sample = X.sample(n=min(4000, len(X)), random_state=42)
configurations = []
for n_groups in (2, 3, 4):
    configurations.extend([
        (f"K-Means k={n_groups}", KMeans(n_clusters=n_groups, n_init=10, random_state=42)),
        (f"GMM k={n_groups}", GaussianMixture(n_components=n_groups, covariance_type="diag", reg_covar=1e-5, n_init=2, random_state=42)),
        (f"Birch k={n_groups}", Birch(n_clusters=n_groups, threshold=0.5)),
    ])
results = []
candidate_pipelines = {}
for name, estimator in configurations:
    candidate = make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler(), estimator)
    candidate.fit(selection_sample)
    labels = candidate.predict(selection_sample)
    if len(np.unique(labels)) < 2:
        continue
    scaled_sample = candidate[:-1].transform(selection_sample)
    proportions = pd.Series(labels).value_counts(normalize=True)
    results.append({
        "modelo": name,
        "silhouette": silhouette_score(scaled_sample, labels),
        "Davies-Bouldin (menor mejor)": davies_bouldin_score(scaled_sample, labels),
        "grupo mas pequeno (%)": 100 * proportions.min(),
    })
    candidate_pipelines[name] = candidate
comparison = pd.DataFrame(results).sort_values(
    ["silhouette", "Davies-Bouldin (menor mejor)"], ascending=[False, True]
).reset_index(drop=True)
if comparison.empty:
    raise ValueError("Ninguna configuracion produjo al menos dos grupos.")
display(comparison.round(3))
best_name = comparison.iloc[0]["modelo"]
best_cluster_model = clone(candidate_pipelines[best_name]).fit(X)
customer_features["cluster"] = best_cluster_model.predict(X)
cluster_profile = customer_features.groupby("cluster")[feature_columns].median()
overall_log = np.log1p(X)
overall_log_median = overall_log.median()
overall_log_std = overall_log.std().replace(0, 1)
profile_scores = (np.log1p(cluster_profile) - overall_log_median) / overall_log_std
feature_phrases = {
    "recency_days": ("compras más recientes", "más tiempo desde la última compra"),
    "frequency": ("frecuencia menor", "frecuencia mayor"),
    "spend_brl": ("gasto menor", "gasto mayor"),
    "category_variety": ("variedad de categorías menor", "variedad de categorías mayor"),
}
cluster_names = {}
cluster_explanations = {}
overall_profile = customer_features[feature_columns].median()
for cluster_id, scores in profile_scores.iterrows():
    salient = scores.loc[scores.abs() >= 0.25].sort_values(key=lambda values: values.abs(), ascending=False).head(2)
    if salient.empty:
        name = "Medianas cercanas al perfil general"
    else:
        phrases = []
        for feature, score in salient.items():
            phrases.append(feature_phrases[feature][int(score > 0)])
        name = " · ".join(phrases).capitalize()
    cluster_names[int(cluster_id)] = name
    size = int(customer_features["cluster"].eq(cluster_id).sum())
    share = size / len(customer_features)
    recency = cluster_profile.loc[cluster_id, "recency_days"]
    overall_recency = overall_profile["recency_days"]
    recency_score = scores["recency_days"]
    recent_comparison = "más reciente que la mediana general" if recency_score <= -0.25 else "menos reciente que la mediana general" if recency_score >= 0.25 else "recencia similar a la mediana general"
    frequency = cluster_profile.loc[cluster_id, "frequency"]
    overall_frequency = overall_profile["frequency"]
    frequency_score = scores["frequency"]
    frequency_comparison = "por encima de la mediana general" if frequency_score >= 0.25 else "por debajo de la mediana general" if frequency_score <= -0.25 else "similar a la mediana general"
    spend = cluster_profile.loc[cluster_id, "spend_brl"]
    overall_spend = overall_profile["spend_brl"]
    spend_score = scores["spend_brl"]
    spend_comparison = "por encima de la mediana general" if spend_score >= 0.25 else "por debajo de la mediana general" if spend_score <= -0.25 else "similar a la mediana general"
    variety = cluster_profile.loc[cluster_id, "category_variety"]
    overall_variety = overall_profile["category_variety"]
    variety_score = scores["category_variety"]
    variety_comparison = "por encima de la mediana general" if variety_score >= 0.25 else "por debajo de la mediana general" if variety_score <= -0.25 else "similar a la mediana general"
    frequency_unit = "pedido" if frequency == 1 else "pedidos"
    variety_unit = "categoría" if variety == 1 else "categorías"
    category_note = " La mediana cero indica que al menos la mitad de los clientes del grupo no tiene categorías registradas en los productos comprados." if variety == 0 else ""
    cluster_explanations[int(cluster_id)] = (
        f"{size:,} clientes ({share:.1%} del conjunto); mediana de recencia {recency:.0f} días ({recent_comparison}: {overall_recency:.0f} días), "
        f"frecuencia mediana {frequency:.0f} {frequency_unit} ({frequency_comparison}: {overall_frequency:.0f}), "
        f"gasto mediano {spend:.2f} BRL ({spend_comparison}: {overall_spend:.2f} BRL) y "
        f"variedad mediana {variety:.0f} {variety_unit} ({variety_comparison}: {overall_variety:.0f}).{category_note}"
    )
cluster_overview = customer_features.groupby("cluster").size().rename("clientes").to_frame()
cluster_overview["porcentaje"] = 100 * cluster_overview["clientes"] / len(customer_features)
cluster_overview["nombre sugerido"] = pd.Series(cluster_names)
print(f"Modelo seleccionado por silhouette en muestra de {len(selection_sample):,}: {best_name}")
display(cluster_overview.join(cluster_profile).round(1))
for cluster_id, name in cluster_names.items():
    print(f"\nResultado sugiere que el Cluster {cluster_id} reúne clientes con el patrón «{name}». {cluster_explanations[cluster_id]} La lectura se refiere a pedidos entregados observados hasta {reference_date.date()}.")
largest_cluster_id = int(cluster_overview["clientes"].idxmax())
largest_cluster_share = cluster_overview.loc[largest_cluster_id, "porcentaje"]
if largest_cluster_share >= 80:
    print(f"Distribución: el Cluster {largest_cluster_id} concentra {largest_cluster_share:.1f}% de los clientes; los otros grupos son minoritarios. Esto indica una partición muy desigual: silhouette seleccionó el modelo por separación interna, pero la distribución también debe considerarse al decidir si los grupos son útiles.")
else:
    print(f"Distribución: el grupo más grande reúne {largest_cluster_share:.1f}% de los clientes; compara los tamaños junto con los perfiles antes de interpretar los grupos.")


## 5. Evaluacion e interpretacion

Cada nombre combina hasta dos variables cuya mediana transformada se aparta al menos 0.25 desviaciones estándar de la mediana general. Una recencia menor significa una última compra más reciente; frecuencia, gasto y variedad mayores indican valores superiores a la mediana del conjunto. La proyección PCA reduce cuatro variables a dos ejes para facilitar la inspección. Los números de grupo son asignaciones del algoritmo; los nombres resumen las diferencias medidas, no son etiquetas originales del dataset.


In [ ]:
projection_sample = X.loc[selection_sample.index]
labels_sample = customer_features.loc[selection_sample.index, "cluster"]
scaled_sample = best_cluster_model[:-1].transform(projection_sample)
projection = PCA(n_components=2, random_state=42).fit_transform(scaled_sample)
plt.figure(figsize=(8, 5))
scatter = plt.scatter(projection[:, 0], projection[:, 1], c=labels_sample, cmap="tab10", alpha=0.65, s=18)
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title(f"PCA exploratoria — {best_name}")
plt.colorbar(scatter, label="Grupo asignado")
plt.tight_layout()
plt.show()
cluster_sizes = customer_features["cluster"].value_counts().sort_index().rename("clientes")
display(cluster_sizes.to_frame().join(pd.Series(cluster_names, name="nombre sugerido")))


## 6. Despliegue: demo Gradio temporal

La demo asigna un perfil con el modelo seleccionado y muestra el nombre descriptivo calculado para ese grupo. La interfaz de Colab es temporal, no un despliegue de producción.


In [ ]:
import gradio as gr

def cluster_demo(recency_days, frequency, spend_brl, category_variety):
    example = pd.DataFrame([[recency_days, frequency, spend_brl, category_variety]], columns=feature_columns)
    group = int(best_cluster_model.predict(example)[0])
    return f"Modelo: {best_name}. Perfil asignado: {cluster_names[group]}. {cluster_explanations[group]}"

demo = gr.Interface(
    fn=cluster_demo,
    inputs=[
        gr.Number(value=60, minimum=0, label="Recencia (dias)"),
        gr.Number(value=2, minimum=1, label="Pedidos observados"),
        gr.Number(value=200, minimum=0, label="Suma de precios (BRL)"),
        gr.Number(value=2, minimum=0, label="Categorias distintas"),
    ],
    outputs=gr.Textbox(label="Perfil y descripción"),
    title="Demo de agrupamiento",
    description=f"Modelo seleccionado por silhouette: {best_name}. Los nombres resumen diferencias observadas frente a la mediana general.",
)
demo.launch(share=False, inline=True, prevent_thread_lock=True)
